In [ ]:
# Hard 7,200-second Kaggle CLI cap includes 300 seconds before this first cell.
import time
SESSION_START = time.monotonic()
SESSION_WATCHDOG_SECONDS = 6900.0
SESSION_CLOSE_MARGIN = 60.0
def session_elapsed():
    return time.monotonic() - SESSION_START


# Burgers pareado: RBF fijo frente a PINNacle Vanilla

Paquete candidato para cinco parejas de seeds nuevas (811–815), con dos Tesla T4 y hasta dos parejas concurrentes. Cada pareja crea una pool Sobol float64 y comparte sus bytes entre los brazos. Cada brazo usa Adam 1e-3, hasta 20,000 updates o 900 GPU-s. Vanilla es MLP tanh 5×100 y no incorpora L-BFGS.

La evaluación Cole–Hopf/Gauss–Hermite de orden 128 en grid 1025×129 se ejecuta en CUDA, solo tras completar todo el entrenamiento y únicamente para checkpoints finales. Cada seed forma una wave de dos procesos score, uno por T4, con cap de 90 s de pared; las cinco waves caben en 450 s. Se conservan checkpoints intermedios para costo, pero no se puntúan.

El artículo PINNacle es contextual: reporta Vanilla con referencia FEM discretizada y protocolo distinto. El cap de sesión solicitado es 7,200 s, incluidos 300 s de arranque previos a la primera celda. El watchdog interno permite 6,900 s desde esa celda y deja 60 s de cierre. La aritmética de peor caso se detalla en manifest.json y budget.json. Este paquete sigue sin aprobación ni autorización de ejecución.


## Reproducibilidad y carga del paquete

La libreta restaura un ZIP autocontenido y verifica su `bundle_id` y SHA-256 de cada archivo. El paquete usa copias estáticas de las fuentes indicadas en `sha256-report.json`; no clona Git ni empaqueta credenciales.


In [ ]:
import base64, io, os, pathlib, sys, zipfile, zlib

BUNDLE_ARCHIVE_B64 = (
    '...'
)
BUNDLE_ID = 'fe21d67d4f6ddffc'
default_work = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').exists() else pathlib.Path.cwd()
WORK_ROOT = pathlib.Path(os.environ.get('KAGGLE_WORKING_DIR', str(default_work))) / ('pikan-' + BUNDLE_ID)
BUNDLE_DIR = WORK_ROOT / 'bundle'
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
archive_bytes = zlib.decompress(base64.b64decode(BUNDLE_ARCHIVE_B64))
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for item in archive.infolist():
        relative = pathlib.PurePosixPath(item.filename)
        if relative.is_absolute() or '..' in relative.parts or '\\' in item.filename:
            raise ValueError('Unsafe bundle path: ' + item.filename)
        target = (BUNDLE_DIR / pathlib.Path(*relative.parts)).resolve()
        if not target.is_relative_to(BUNDLE_DIR.resolve()):
            raise ValueError('Bundle path escaped destination: ' + item.filename)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(item))
sys.path.insert(0, str(BUNDLE_DIR))
import importlib.util
_runtime_path = BUNDLE_DIR / 'runtime.py'
if not _runtime_path.is_file():
    raise FileNotFoundError(f'Bundled root runtime is missing: {_runtime_path}')
_runtime_spec = importlib.util.spec_from_file_location('af_pikans_bundle_bootstrap_runtime', _runtime_path)
if _runtime_spec is None or _runtime_spec.loader is None:
    raise ImportError(f'Could not load bundled root runtime: {_runtime_path}')
_runtime_module = importlib.util.module_from_spec(_runtime_spec)
sys.modules[_runtime_spec.name] = _runtime_module
_runtime_spec.loader.exec_module(_runtime_module)
SPEC = _runtime_module.verify_bundle(BUNDLE_DIR)
assert SPEC['bundle_id'].startswith(BUNDLE_ID)
RUNS_DIR = WORK_ROOT / 'runs'
ANALYSIS_DIR = WORK_ROOT / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print('Bundle verified:', SPEC['bundle_id'], 'source files:', len(SPEC['files']))


## 2. Dependencias y versiones

PyTorch/CUDA se comprueban contra Kaggle. Las dependencias restantes se instalan
solo si falta un pin o su versión difiere. Panel y PyVista no se instalan en el
entrenamiento porque estos dominios 1D no los necesitan.


In [ ]:
import importlib.metadata, subprocess, sys

DEPENDENCIES = []
missing_or_different = []
for requirement in DEPENDENCIES:
    name, expected = requirement.split('==', 1)
    try:
        installed = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != expected:
        missing_or_different.append(requirement)
if missing_or_different:
    command = [sys.executable, '-m', 'pip', 'install',
               '--disable-pip-version-check', *missing_or_different]
    print('Installing pinned dependencies:', ', '.join(missing_or_different), flush=True)
    subprocess.run(command, check=True, timeout=300)
else:
    print('Pinned experiment dependencies already match the manifest.')


## 3. Protocolo y decisiones de parámetros

El manifiesto es la fuente de verdad para condiciones, arquitectura, muestreo,
semillas, pasos, tiempo, diagnóstico y versiones. Las tablas reportan valores
calculados desde artefactos; las decisiones completas se conservan junto a resultados.


In [ ]:
from IPython.display import Markdown, display
import json, pandas as pd
protocol=json.loads((BUNDLE_DIR/'manifest.json').read_text(encoding='utf-8'))
display(Markdown(protocol['parameter_decisions_markdown']))
display(pd.json_normalize(protocol['training']).T)


## Preflight: dos Tesla T4

Dos workers de seed se ejecutan concurrentemente; cada worker ve una T4, entrena RBF y Vanilla secuencialmente y comparte pools exactas. Los dos workers usan los slots 0 y 1. El scorer final también inicia dos procesos paralelos con una GPU visible cada uno. La telemetría registra el slot y el nombre reportado por CUDA.


In [ ]:
import platform, subprocess, torch
import pandas as pd
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable; the approved T4x2 campaign did not start.')
visible_gpus = torch.cuda.device_count()
gpu_names = [torch.cuda.get_device_name(index) for index in range(visible_gpus)]
if visible_gpus < 2:
    raise RuntimeError(f'T4x2 was requested, but only {visible_gpus} CUDA devices are visible: {gpu_names}')
if any('T4' not in name.upper() for name in gpu_names[:2]):
    raise RuntimeError(f'The requested accelerator is 2xT4; visible devices are {gpu_names[:2]}')
GPU_SLOTS = ['0', '1']
hardware = pd.DataFrame([
    {'logical_gpu': i, 'name': gpu_names[i],
     'memory_total_mib': round(torch.cuda.get_device_properties(i).total_memory / 2**20),
     'parallel_seed_slot': GPU_SLOTS[i]} for i in range(2)
])
display(hardware)
display(pd.DataFrame([{'python': platform.python_version(), 'pytorch': torch.__version__,
                       'cuda_runtime': torch.version.cuda, 'visible_gpu_count': visible_gpus}]))
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'],
               check=True, text=True)


In [ ]:
import copy, json, os, subprocess, sys, tempfile
from pathlib import Path

manifest = json.loads((BUNDLE_DIR / 'manifest.json').read_text(encoding='utf-8'))
if manifest.get('budget', {}).get('approved') is not True:
    raise RuntimeError('Draft only: human budget approval is required before smoke or training.')
if manifest.get('research_status', {}).get('execution_authorized') is not True:
    raise RuntimeError('Draft only: execution authorization is required before smoke or training.')
smoke_guard = WORK_ROOT / 'smoke-started.json'
if smoke_guard.exists():
    raise RuntimeError('Smoke is single-use under this budget; inspect its prior artifacts before any retry.')
smoke_root = WORK_ROOT / 'smoke'
smoke_root.mkdir(parents=True, exist_ok=True)
smoke_attempt = Path(tempfile.mkdtemp(prefix='attempt-', dir=smoke_root))
smoke_manifest = copy.deepcopy(manifest)
smoke_manifest['training']['seeds'] = [manifest['training']['seeds'][0]]
smoke_manifest['training']['seconds_per_arm'] = int(smoke_manifest['training']['smoke_seconds_per_arm'])
smoke_manifest_path = smoke_attempt / 'manifest-smoke.json'
smoke_manifest_path.write_text(json.dumps(smoke_manifest, indent=2), encoding='utf-8')
smoke_run = smoke_attempt / f"seed-{manifest['training']['seeds'][0]}"
smoke_guard.write_text(json.dumps({'status': 'started', 'run_dir': str(smoke_run)}), encoding='utf-8')
smoke_env = os.environ.copy()
smoke_env.update({
    'HARNESS_MANIFEST': str(smoke_manifest_path),
    'HARNESS_RUN_DIR': str(smoke_run),
    'HARNESS_SEED': str(manifest['training']['seeds'][0]),
    'HARNESS_GPU_SLOT': GPU_SLOTS[0],
    'CUDA_VISIBLE_DEVICES': GPU_SLOTS[0],
    'PYTHONPATH': str(BUNDLE_DIR),
    'PYTHONUNBUFFERED': '1',
})
if session_elapsed() > 300.0:
    raise RuntimeError('Setup exceeded its 300-second allowance before smoke.')
smoke_process = subprocess.run(
    [sys.executable, str(BUNDLE_DIR / SPEC['entrypoint'])],
    cwd=BUNDLE_DIR, env=smoke_env, check=False, timeout=240,
)
if smoke_process.returncode != 0:
    raise RuntimeError(f'Paired smoke failed with exit code {smoke_process.returncode}; do not retry automatically.')
smoke_guard.write_text(json.dumps({'status': 'completed', 'run_dir': str(smoke_run)}), encoding='utf-8')
smoke_status = json.loads((smoke_run / 'problem_status.json').read_text(encoding='utf-8'))
if smoke_status.get('shared_pool_exact') is not True:
    raise RuntimeError('Smoke did not preserve exact pairwise pool identity.')
print('Smoke completed; no evaluator or held-out field score was run.')


## Pools compartidas por pareja

El wrapper genera una sola pool Sobol por seed, guarda sus cinco arrays y sus hashes, y confirma igualdad exacta después de cada brazo. El smoke no puntúa error de campo. La inspección detallada de cobertura se conserva en `paired_pool_manifest.json` por seed.


In [ ]:
# No se generan pools antes de la ejecución.
# Cada worker las genera una vez para compartirlas entre ambos brazos.
print('Pooled sampling is performed once per paired seed by the reviewed entrypoint.')


## Smoke y entrenamiento por pares

Primero se ejecuta un único smoke de 60 GPU-s por brazo sin scoring ni selección. Si termina dentro del cap de 240 s de pared, `run_bundle` procesa las cinco parejas en tres waves, con hasta dos parejas concurrentes y 1,920 s de tope por pareja. Cada brazo para en 20,000 Adam updates o 900 GPU-s. Un fallo se conserva; no se reemplaza seed.


In [ ]:
import importlib.util, json, sys
runtime_path = BUNDLE_DIR / 'src' / 'research_pipeline' / 'remote' / 'runtime.py'
if not runtime_path.is_file():
    raise FileNotFoundError(f'Bundled remote runtime is missing: {runtime_path}')
runtime_spec = importlib.util.spec_from_file_location('af_pikans_remote_runtime', runtime_path)
if runtime_spec is None or runtime_spec.loader is None:
    raise ImportError(f'Could not load bundled remote runtime: {runtime_path}')
runtime_module = importlib.util.module_from_spec(runtime_spec)
sys.modules[runtime_spec.name] = runtime_module
runtime_spec.loader.exec_module(runtime_module)
run_bundle = runtime_module.run_bundle

manifest = json.loads((BUNDLE_DIR / 'manifest.json').read_text(encoding='utf-8'))
if manifest.get('budget', {}).get('approved') is not True or manifest.get('research_status', {}).get('execution_authorized') is not True:
    raise RuntimeError('Draft only: signed budget and execution authorization are required.')
smoke_status = json.loads((WORK_ROOT / 'smoke-started.json').read_text(encoding='utf-8'))
if smoke_status.get('status') != 'completed':
    raise RuntimeError('The single paired smoke must finish before Tier 2.')
remaining = 3 * 1920 + 5 * 90 + 90
if session_elapsed() + remaining > SESSION_WATCHDOG_SECONDS - SESSION_CLOSE_MARGIN:
    raise RuntimeError('Insufficient guarded time for all Tier 2 pairs, endpoint scoring, and artifact flush.')
campaign_state = run_bundle(BUNDLE_DIR, RUNS_DIR, max_parallel=2, gpu_slots=GPU_SLOTS)
display(pd.DataFrame([{'seed': seed, 'status': status} for seed, status in sorted(campaign_state['seeds'].items())]))
if any(status != 'completed' for status in campaign_state['seeds'].values()):
    raise RuntimeError('Incomplete paired campaign; keep all failures and do not replace seeds.')


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
import pandas as pd
import torch
from IPython.display import display

manifest = json.loads((BUNDLE_DIR / 'manifest.json').read_text(encoding='utf-8'))
expected_seeds = manifest['training']['seeds']
seed_dirs = sorted(RUNS_DIR.glob('seed-*'))
if [int(path.name.split('-')[1]) for path in seed_dirs] != expected_seeds:
    raise RuntimeError('Endpoint scoring requires all five fixed seeds; no seed may be dropped.')
SCORE_WAVE_SECONDS = 90.0
SCORE_WAVES = len(expected_seeds)
ARTIFACT_FLUSH_SECONDS = 90.0
CLOSE_MARGIN_SECONDS = SESSION_CLOSE_MARGIN
score_root = ANALYSIS_DIR / 'scores'
score_root.mkdir(parents=True, exist_ok=True)
ledger_path = ANALYSIS_DIR / 'endpoint-score-ledger.json'
score_ledger = {'status': 'running', 'score_wave_cap_seconds': SCORE_WAVE_SECONDS, 'waves': []}
remaining_score_wall = SCORE_WAVES * SCORE_WAVE_SECONDS
if session_elapsed() + remaining_score_wall + ARTIFACT_FLUSH_SECONDS + CLOSE_MARGIN_SECONDS > SESSION_WATCHDOG_SECONDS:
    raise RuntimeError('Insufficient watchdog budget for all endpoint score waves and artifact flush.')

records = []
scorer = BUNDLE_DIR / 'burgers_cuda_score.py'
for seed_index, seed in enumerate(expected_seeds):
    remaining_after = (SCORE_WAVES - seed_index) * SCORE_WAVE_SECONDS
    if session_elapsed() + remaining_after + ARTIFACT_FLUSH_SECONDS + CLOSE_MARGIN_SECONDS > SESSION_WATCHDOG_SECONDS:
        score_ledger['status'] = 'incomplete_time_budget'
        ledger_path.write_text(json.dumps(score_ledger, indent=2), encoding='utf-8')
        raise RuntimeError('Endpoint scoring cannot finish within the guarded session budget.')
    jobs = []
    for arm_index, arm in enumerate(('rbf', 'vanilla')):
        relative = 'burgers-rbf' if arm == 'rbf' else 'burgers-vanilla'
        checkpoint = RUNS_DIR / f'seed-{seed}' / relative / 'checkpoint-final.pt'
        if not checkpoint.is_file():
            raise RuntimeError(f'Missing endpoint checkpoint: {checkpoint}')
        out = score_root / f'seed-{seed}' / f'{arm}-endpoint'
        out.mkdir(parents=True, exist_ok=True)
        stdout = (out / 'stdout.log').open('w', encoding='utf-8')
        stderr = (out / 'stderr.log').open('w', encoding='utf-8')
        env = os.environ.copy()
        env.update({
            'CUDA_VISIBLE_DEVICES': GPU_SLOTS[arm_index],
            'PYTHONPATH': str(BUNDLE_DIR),
            'PYTHONUNBUFFERED': '1',
        })
        command = [sys.executable, str(scorer), '--checkpoint', str(checkpoint),
                   '--arm', arm, '--seed', str(seed), '--out', str(out)]
        process = subprocess.Popen(command, cwd=BUNDLE_DIR, env=env, stdout=stdout, stderr=stderr)
        jobs.append((arm, out, process, stdout, stderr))
    wave_start = time.monotonic()
    while any(job[2].poll() is None for job in jobs):
        if (time.monotonic() - wave_start > SCORE_WAVE_SECONDS or
                session_elapsed() >= SESSION_WATCHDOG_SECONDS - SESSION_CLOSE_MARGIN):
            for arm, out, process, stdout, stderr in jobs:
                if process.poll() is None:
                    process.terminate()
                stdout.close()
                stderr.close()
            score_ledger['status'] = 'incomplete_score_wave_timeout'
            score_ledger['waves'].append({'seed': seed, 'status': 'timeout'})
            ledger_path.write_text(json.dumps(score_ledger, indent=2), encoding='utf-8')
            raise RuntimeError(f'CUDA endpoint score wave for seed {seed} exceeded its 90-second allowance.')
        time.sleep(1.0)
    wave_rows = []
    for arm, out, process, stdout, stderr in jobs:
        code = process.wait()
        stdout.close()
        stderr.close()
        if code != 0:
            score_ledger['status'] = 'incomplete_score_process_failure'
            score_ledger['waves'].append({'seed': seed, 'arm': arm, 'returncode': code})
            ledger_path.write_text(json.dumps(score_ledger, indent=2), encoding='utf-8')
            raise RuntimeError(f'CUDA endpoint scorer failed for seed {seed}, arm {arm}.')
        report = json.loads((out / 'score.json').read_text(encoding='utf-8'))
        if not report.get('endpoint_only') or 't4' not in report.get('scoring_device', '').lower():
            raise RuntimeError('Expected endpoint-only scoring on the assigned CUDA T4.')
        metrics = report['metrics']
        row = {'seed': seed, 'arm': arm, 'checkpoint': 'checkpoint-final.pt',
               'wall_score_wave_seconds': time.monotonic() - wave_start,
               'device': report['scoring_device'],
               'evaluation_gpu_seconds': report['evaluation_gpu_seconds_estimate'],
               'field_relative_l2': float(metrics['field_relative_l2']),
               'field_rmse_abs': float(metrics['field_rmse_abs']),
               'residual_rmse_abs': float(metrics['residual_rmse_abs']),
               'scientific_accepted': False}
        wave_rows.append(row)
        records.append(row)
    score_ledger['waves'].append({'seed': seed, 'status': 'scored', 'rows': wave_rows})
    ledger_path.write_text(json.dumps(score_ledger, indent=2), encoding='utf-8')

if session_elapsed() + ARTIFACT_FLUSH_SECONDS + CLOSE_MARGIN_SECONDS > SESSION_WATCHDOG_SECONDS:
    score_ledger['status'] = 'incomplete_artifact_flush_budget'
    ledger_path.write_text(json.dumps(score_ledger, indent=2), encoding='utf-8')
    raise RuntimeError('Insufficient guarded time for endpoint summary artifacts.')
scores = pd.DataFrame(records).sort_values(['seed', 'arm'])
scores.to_csv(ANALYSIS_DIR / 'paired-endpoint-scores.csv', index=False)
pairs = scores.pivot(index='seed', columns='arm', values='field_relative_l2').reindex(expected_seeds)
if len(pairs) != 5 or pairs.isna().any().any():
    raise RuntimeError('Five complete final endpoint pairs are required.')
score_device = torch.device('cuda:0')
log_ratios = torch.log(torch.as_tensor(
    pairs['rbf'].to_numpy() / pairs['vanilla'].to_numpy(), dtype=torch.float64, device=score_device))
rng = torch.Generator(device=score_device).manual_seed(20260927)
bootstrap_indices = torch.randint(0, len(log_ratios), (10000, len(log_ratios)), generator=rng, device=score_device)
bootstrap = log_ratios[bootstrap_indices].mean(dim=1)
bootstrap_interval = torch.exp(torch.quantile(bootstrap, torch.tensor([0.025, 0.975], dtype=torch.float64, device=score_device)))
summary = {
    'n_pairs': 5,
    'primary_endpoint': 'final checkpoint only, scored after all training; no intermediate checkpoint scores',
    'summary_device': torch.cuda.get_device_name(0),
    'geometric_mean_rbf_over_vanilla': float(torch.exp(log_ratios.mean()).item()),
    'paired_log_ratio_values': {str(seed): float(value) for seed, value in zip(expected_seeds, log_ratios.detach().cpu().tolist())},
    'bootstrap_95_percentile_interval_ratio': bootstrap_interval.detach().cpu().tolist(),
    'proposed_noninferiority_margin': 1.10,
    'margin_signed': False,
    'decision': 'descriptive_only_pending_signed preregistration',
    'scores_used_for_tuning': False,
    'scientific_accepted': False,
}
(ANALYSIS_DIR / 'paired-primary-summary.json').write_text(json.dumps(summary, indent=2, allow_nan=False) + '\\n', encoding='utf-8')
score_ledger['status'] = 'complete'
score_ledger['total_wall_seconds'] = sum(wave['rows'][0]['wall_score_wave_seconds'] for wave in score_ledger['waves'] if wave.get('status') == 'scored')
ledger_path.write_text(json.dumps(score_ledger, indent=2), encoding='utf-8')
display(scores)
display(pd.DataFrame([summary]))


## Score confirmatorio de endpoints

Se puntúa solo `checkpoint-final.pt`, para ambos brazos de cada una de las cinco seeds y después de concluir todo el entrenamiento. El scorer independiente calcula referencia, derivadas, malla, residuales y métricas en CUDA. Los scores no se usan para tuning. El bootstrap descriptivo se predefine en el notebook; el margen 1.10 sigue sin firma y no genera una decisión de aceptación.


In [ ]:
from IPython.display import display
import json, pandas as pd
scores = pd.read_csv(ANALYSIS_DIR / 'paired-endpoint-scores.csv')
summary = json.loads((ANALYSIS_DIR / 'paired-primary-summary.json').read_text(encoding='utf-8'))
display(scores)
display(pd.DataFrame([summary]))


## Comparación de endpoints

Este paquete conserva checkpoints intermedios para documentar costo, pero no los puntúa. El gráfico siguiente compara los diez scores finales pareados; no representa una curva de aprendizaje ni selecciona checkpoints.


In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display
scores = pd.read_csv(ANALYSIS_DIR / 'paired-endpoint-scores.csv')
figure = px.scatter(scores, x='arm', y='field_relative_l2', color='seed', log_y=True,
                    labels={'arm': 'arm', 'field_relative_l2': 'relative L2'},
                    title='Final endpoint scores (each point scored on CUDA)')
figure.write_html(ANALYSIS_DIR / 'paired-endpoint-scores.html', include_plotlyjs='cdn')
display(figure)


## Campos evaluados

Cada scorer produce arrays de predicción, referencia, error y residual sobre la malla CUDA común. Se guardan en archivos NPZ para inspección posterior; las conversiones a CPU ocurren únicamente al serializar esos arrays.


In [ ]:
from IPython.display import IFrame, display
import json
from pathlib import Path
for seed in json.loads((BUNDLE_DIR / 'manifest.json').read_text(encoding='utf-8'))['training']['seeds']:
    for arm in ('rbf', 'vanilla'):
        archive = ANALYSIS_DIR / 'scores' / f'seed-{seed}' / f'{arm}-endpoint' / 'validation.npz'
        if archive.is_file():
            print(f'seed {seed}, {arm}: final CUDA-scored prediction/reference/error/residual at {archive}')


## Cierre y hashes de resultados

El ZIP conserva el bundle, configuración, pools, checkpoints, costo, telemetría y score confirmatorio. Ninguna cifra se marca como aceptada. El paquete exige aprobación humana ligada a este hash antes de cualquier ejecución remota.


In [ ]:
import json
from src.research_pipeline.remote.postprocess import create_results_archive
manifest = json.loads((BUNDLE_DIR / 'manifest.json').read_text(encoding='utf-8'))
(ANALYSIS_DIR / 'parameter-decisions.md').write_text(manifest['parameter_decisions_markdown'], encoding='utf-8')
archive_path = WORK_ROOT / 'burgers-rbf-vanilla-paired-results.zip'
archive_result = create_results_archive(WORK_ROOT, ANALYSIS_DIR, archive_path)
print('Paired result archive:', archive_result['archive'], '; hashed files:', archive_result['file_count'])
print('Draft protocol only; no scientific acceptance or SOTA claim is asserted.')
